In [1]:
import os
os.chdir('/Users/angelinagupta/Desktop/sem 6/biomedical-nlp-project-beta')

In [2]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sklearn-crfsuite', 'seqeval'], check=True)
print('Dependencies ready.')

Dependencies ready.


In [3]:
import pandas as pd
import numpy as np
import json
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from itertools import product

LABEL_MAP = {0: 'O', 1: 'B-Disease', 2: 'I-Disease'}
PARQUET_BASE = 'hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease'

df_train = pd.read_parquet(f'{PARQUET_BASE}/train/0000.parquet')
df_val   = pd.read_parquet(f'{PARQUET_BASE}/validation/0000.parquet')
df_test  = pd.read_parquet(f'{PARQUET_BASE}/test/0000.parquet')

print(f'Train : {len(df_train)} sentences')
print(f'Val   : {len(df_val)}   sentences')
print(f'Test  : {len(df_test)}  sentences')

Train : 5433 sentences
Val   : 924   sentences
Test  : 941  sentences


---
## What is a CRF (Conditional Random Field)?

Imagine you're tagging a sentence word-by-word, deciding for each word: is it `O`, `B-Disease`, or `I-Disease`?

A **naive classifier** (like logistic regression applied per token) looks at each word in isolation and picks the most likely tag. That's fast, but it ignores a critical constraint: in BIO tagging, `I-Disease` should only follow `B-Disease` or another `I-Disease` — never an `O` tag.

A **CRF (Conditional Random Field)** fixes this by modelling the **entire label sequence jointly**. Instead of maximising $P(y_i \mid x_i)$ for each token independently, it maximises:

$$P(\mathbf{y} \mid \mathbf{x}) = \frac{1}{Z(\mathbf{x})} \exp\!\left(\sum_{i} \sum_{k} \lambda_k f_k(y_{i-1}, y_i, \mathbf{x}, i)\right)$$

where $f_k$ are **feature functions** and $\lambda_k$ are learned weights.

### Two types of features in a CRF

| Feature type | What it captures | Example |
|---|---|---|
| **State features** | Relationship between the current word's properties and its tag | `word.lower() = 'hepatitis' → B-Disease` |
| **Transition features** | Which tag is likely to follow which tag | `B-Disease → I-Disease` (high weight), `O → I-Disease` (large negative weight) |

The CRF learns both simultaneously during training using gradient-based optimisation (here: L-BFGS). At inference it uses **Viterbi decoding** to find the globally optimal tag sequence — not just the locally best tag at each step.

### Why feature engineering matters

A CRF is only as good as the features you feed it. Unlike deep learning, it cannot learn representations from raw pixels or characters — you have to hand-code the signals you believe are useful. For biomedical NER:
- Suffixes (`-itis`, `-oma`) are strong lexical signals
- Capitalisation patterns distinguish abbreviations (ALL-CAPS) from normal words
- Context (previous/next word) tells the model about surrounding structure
- Hyphenation is common in medical compound terms

---
## Feature Engineering

In [4]:
MEDICAL_SUFFIXES = (
    'itis', 'osis', 'emia', 'oma', 'pathy',
    'plasia', 'ectomy', 'uria', 'trophy', 'sclerosis'
)

def has_medical_suffix(word):
    w = word.lower()
    return any(w.endswith(s) for s in MEDICAL_SUFFIXES)


def word2features(sent, i):
    """Return a feature dict for token at position i in sentence `sent`."""
    word = sent[i]
    wl   = word.lower()

    features = {
        # --- Current token ---
        'bias'              : 1.0,
        'word.lower'        : wl,
        'word.isupper'      : word.isupper(),
        'word.istitle'      : word.istitle(),
        'word.isdigit'      : word.isdigit(),
        'word.len'          : len(word),
        'word[-3:]'         : wl[-3:],
        'word[-5:]'         : wl[-5:],
        'word[:3]'          : wl[:3],       # prefix
        'has_hyphen'        : '-' in word,
        'has_digit'         : any(c.isdigit() for c in word),
        'has_medical_suffix': has_medical_suffix(word),
    }

    # --- Previous token (i-1) ---
    if i == 0:
        features['BOS'] = True
    else:
        pw = sent[i - 1]
        pwl = pw.lower()
        features.update({
            '-1:word.lower'        : pwl,
            '-1:word.isupper'      : pw.isupper(),
            '-1:word.istitle'      : pw.istitle(),
            '-1:word[-3:]'         : pwl[-3:],
            '-1:has_medical_suffix': has_medical_suffix(pw),
        })

    # --- Next token (i+1) ---
    if i == len(sent) - 1:
        features['EOS'] = True
    else:
        nw = sent[i + 1]
        nwl = nw.lower()
        features.update({
            '+1:word.lower'        : nwl,
            '+1:word.isupper'      : nw.isupper(),
            '+1:word.istitle'      : nw.istitle(),
            '+1:word[-3:]'         : nwl[-3:],
            '+1:has_medical_suffix': has_medical_suffix(nw),
        })

    # --- Bigram: current + next (lowercased) ---
    if i < len(sent) - 1:
        features['bigram'] = wl + '_' + sent[i + 1].lower()

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(tags):
    """Convert integer or string tag list to string BIO labels."""
    return [
        LABEL_MAP.get(t, 'O') if isinstance(t, (int, np.integer)) else str(t)
        for t in tags
    ]


# Quick sanity check
sample_sent = list(df_train['tokens'].iloc[0])
sample_feat = sent2features(sample_sent)
print('Sentence:', sample_sent[:6])
print('\nFeatures for token 0:')
for k, v in sample_feat[0].items():
    print(f'  {k:30s} = {v}')

Sentence: ['Identification', 'of', 'APC2', ',', 'a', 'homologue']

Features for token 0:
  bias                           = 1.0
  word.lower                     = identification
  word.isupper                   = False
  word.istitle                   = True
  word.isdigit                   = False
  word.len                       = 14
  word[-3:]                      = ion
  word[-5:]                      = ation
  word[:3]                       = ide
  has_hyphen                     = False
  has_digit                      = False
  has_medical_suffix             = False
  BOS                            = True
  +1:word.lower                  = of
  +1:word.isupper                = False
  +1:word.istitle                = False
  +1:word[-3:]                   = of
  +1:has_medical_suffix          = False
  bigram                         = identification_of


---
## Convert Data to Feature / Label Sequences

In [5]:
def df_to_XY(df):
    X, y = [], []
    for _, row in df.iterrows():
        tokens = list(row['tokens'])
        tags   = list(row['ner_tags'])
        if len(tokens) == 0:
            continue
        X.append(sent2features(tokens))
        y.append(sent2labels(tags))
    return X, y


print('Converting train...')
X_train, y_train = df_to_XY(df_train)
print('Converting val...')
X_val,   y_val   = df_to_XY(df_val)
print('Converting test...')
X_test,  y_test  = df_to_XY(df_test)

print(f'\nTrain: {len(X_train)} seqs  |  Val: {len(X_val)}  |  Test: {len(X_test)}')
print(f'Labels in y_train[0]: {y_train[0][:10]}')

# Sanity: ensure all seq pairs have matching lengths
for split_name, X, y in [('train', X_train, y_train),
                           ('val',   X_val,   y_val),
                           ('test',  X_test,  y_test)]:
    mismatches = [(i, len(X[i]), len(y[i])) for i in range(len(X)) if len(X[i]) != len(y[i])]
    if mismatches:
        print(f'MISMATCH in {split_name}: {mismatches[:5]}')
    else:
        print(f'{split_name}: all sequence lengths match ✓')

Converting train...
Converting val...
Converting test...

Train: 5432 seqs  |  Val: 923  |  Test: 940
Labels in y_train[0]: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Disease', 'I-Disease']
train: all sequence lengths match ✓
val: all sequence lengths match ✓
test: all sequence lengths match ✓


---
## Train CRF (Default Hyperparameters)

In [6]:
crf = sklearn_crfsuite.CRF(
    algorithm      = 'lbfgs',
    c1             = 0.1,
    c2             = 0.1,
    max_iterations = 100,
    all_possible_transitions = True
)

crf.fit(X_train, y_train)
print('Training complete.')

labels = ['B-Disease', 'I-Disease']  # exclude O for entity-level eval

y_pred_default = crf.predict(X_test)

prec_d = precision_score(y_test, y_pred_default)
rec_d  = recall_score(y_test, y_pred_default)
f1_d   = f1_score(y_test, y_pred_default)

print(f'\n[Default c1=0.1, c2=0.1]')
print(f'  Precision : {prec_d:.4f}')
print(f'  Recall    : {rec_d:.4f}')
print(f'  F1        : {f1_d:.4f}')
print('\nDetailed Report:')
print(classification_report(y_test, y_pred_default))

Training complete.

[Default c1=0.1, c2=0.1]
  Precision : 0.8310
  Recall    : 0.7375
  F1        : 0.7815

Detailed Report:
              precision    recall  f1-score   support

     Disease       0.83      0.74      0.78       960

   micro avg       0.83      0.74      0.78       960
   macro avg       0.83      0.74      0.78       960
weighted avg       0.83      0.74      0.78       960



---
## Top 20 Most Informative Features

After training, the CRF has a weight $\lambda_k$ for every (feature, label) pair — these are **state features**. A large positive weight means "this feature is strong evidence for this label". We inspect the top weights for `B-Disease` and `I-Disease` to understand what the model learned.

In [7]:
from collections import defaultdict

def top_state_features(crf_model, label, n=20):
    """Return top-n (feature, weight) pairs for a given label."""
    state_feats = defaultdict(float)
    for (attr, lbl), weight in crf_model.state_features_.items():
        if lbl == label:
            state_feats[attr] = weight
    return sorted(state_feats.items(), key=lambda x: -x[1])[:n]


for label in ['B-Disease', 'I-Disease']:
    print(f'\n=== Top 20 features for [{label}] ===')
    print(f'  {"Feature":<45}  Weight')
    print('  ' + '-' * 55)
    for feat, w in top_state_features(crf, label):
        print(f'  {feat:<45}  {w:+.4f}')


=== Top 20 features for [B-Disease] ===
  Feature                                        Weight
  -------------------------------------------------------
  BOS                                            +4.4730
  word.isupper                                   +3.8602
  word[:3]:obe                                   +3.8001
  bigram:isolated_aniridia                       +3.6676
  bigram:severe_von                              +3.6316
  word[:3]:edm                                   +3.4997
  bigram:deficiency_of                           +3.3992
  word[:3]:goi                                   +3.3187
  word[-3:]:mas                                  +3.2936
  word.lower:neurodegeneration                   +2.8069
  word.lower:hypopigmentation                    +2.7945
  bigram:apc_gene                                +2.7942
  bigram:frda_gene                               +2.6854
  bigram:apc_mutations                           +2.5707
  word.lower:demyelination                     

---
## Error Analysis — 10 Misclassified Examples

In [8]:
print('10 sentences with prediction errors\n')
print('  ** marks mismatch tokens:  token [TRUE | PRED]\n')

shown = 0
for i, (true_seq, pred_seq) in enumerate(zip(y_test, y_pred_default)):
    if true_seq == pred_seq:
        continue
    tokens = list(df_test.iloc[i]['tokens'])
    print(f'--- Sentence {i} ---')
    parts = []
    for tok, t, p in zip(tokens, true_seq, pred_seq):
        if t != p:
            parts.append(f'**{tok}**[{t}|{p}]')
        else:
            parts.append(f'{tok}[{t}]')
    print(' '.join(parts))
    print()
    shown += 1
    if shown >= 10:
        break

10 sentences with prediction errors

  ** marks mismatch tokens:  token [TRUE | PRED]

--- Sentence 3 ---
By[O] analysing[O] tumour[B-Disease] DNA[O] from[O] patients[O] with[O] sporadic[B-Disease] T[I-Disease] -[I-Disease] cell[I-Disease] prolymphocytic[I-Disease] leukaemia[I-Disease] ([O] **T**[B-Disease|O] **-**[I-Disease|O] **PLL**[I-Disease|O] )[O] ,[O] a[O] rare[O] clonal[B-Disease] malignancy[I-Disease] with[O] similarities[O] to[O] a[O] **mature**[B-Disease|O] **T**[I-Disease|O] **-**[I-Disease|O] **cell**[I-Disease|O] **leukaemia**[I-Disease|B-Disease] seen[O] in[O] A[B-Disease] -[I-Disease] T[I-Disease] ,[O] we[O] demonstrate[O] a[O] high[O] frequency[O] of[O] ATM[O] mutations[O] in[O] **T**[B-Disease|O] **-**[I-Disease|O] **PLL**[I-Disease|O] .[O]

--- Sentence 7 ---
Two[O] of[O] seventeen[O] mutated[O] **T**[B-Disease|O] **-**[I-Disease|O] **PLL**[I-Disease|O] samples[O] had[O] a[O] previously[O] reported[O] A[B-Disease] -[I-Disease] T[I-Disease] allele[O] .[O]

--- Sentenc

---
## Hyperparameter Tuning — Grid Search on Validation Set

The regularisation parameters `c1` (L1) and `c2` (L2) control how aggressively the model penalises large weights:
- **c1 (L1)** encourages sparsity — many feature weights get pushed to exactly 0 (feature selection effect)
- **c2 (L2)** prevents any single weight from growing too large (stability)

We use the **validation set** (not the test set) to pick the best combination, then report the test score once.

In [9]:
c1_values = [0.01, 0.1, 0.5]
c2_values = [0.01, 0.1, 0.5]

best_f1     = -1
best_params = {}
grid_results = []

total = len(c1_values) * len(c2_values)
run   = 0

for c1_val, c2_val in product(c1_values, c2_values):
    run += 1
    crf_gs = sklearn_crfsuite.CRF(
        algorithm      = 'lbfgs',
        c1             = c1_val,
        c2             = c2_val,
        max_iterations = 100,
        all_possible_transitions = True
    )
    crf_gs.fit(X_train, y_train)
    y_pred_val = crf_gs.predict(X_val)
    val_f1     = f1_score(y_val, y_pred_val)

    grid_results.append({'c1': c1_val, 'c2': c2_val, 'val_f1': val_f1})
    print(f'  [{run:2d}/{total}] c1={c1_val:.2f}  c2={c2_val:.2f}  →  val F1={val_f1:.4f}')

    if val_f1 > best_f1:
        best_f1     = val_f1
        best_params = {'c1': c1_val, 'c2': c2_val}

print(f'\nBest val F1 = {best_f1:.4f}  →  {best_params}')

  [ 1/9] c1=0.01  c2=0.01  →  val F1=0.8071
  [ 2/9] c1=0.01  c2=0.10  →  val F1=0.8003
  [ 3/9] c1=0.01  c2=0.50  →  val F1=0.8050
  [ 4/9] c1=0.10  c2=0.01  →  val F1=0.8092
  [ 5/9] c1=0.10  c2=0.10  →  val F1=0.8084
  [ 6/9] c1=0.10  c2=0.50  →  val F1=0.8003
  [ 7/9] c1=0.50  c2=0.01  →  val F1=0.8013
  [ 8/9] c1=0.50  c2=0.10  →  val F1=0.8029
  [ 9/9] c1=0.50  c2=0.50  →  val F1=0.7875

Best val F1 = 0.8092  →  {'c1': 0.1, 'c2': 0.01}


---
## Retrain with Best Hyperparameters → Final Test Evaluation

In [10]:
crf_best = sklearn_crfsuite.CRF(
    algorithm      = 'lbfgs',
    c1             = best_params['c1'],
    c2             = best_params['c2'],
    max_iterations = 200,   # more iterations for final model
    all_possible_transitions = True
)

# Train on train + val combined for the final model
crf_best.fit(X_train + X_val, y_train + y_val)

y_pred_best = crf_best.predict(X_test)

# Lengths must match — verify
mismatches = [(i, len(y_test[i]), len(y_pred_best[i]))
              for i in range(len(y_test))
              if len(y_test[i]) != len(y_pred_best[i])]
assert len(mismatches) == 0, f'Length mismatches: {mismatches[:5]}'

prec_best = precision_score(y_test, y_pred_best)
rec_best  = recall_score(y_test, y_pred_best)
f1_best   = f1_score(y_test, y_pred_best)

print(f'[Best CRF  c1={best_params["c1"]}  c2={best_params["c2"]}  train+val]')
print(f'  Precision : {prec_best:.4f}')
print(f'  Recall    : {rec_best:.4f}')
print(f'  F1        : {f1_best:.4f}')
print()
print('Detailed Classification Report:')
print(classification_report(y_test, y_pred_best))

[Best CRF  c1=0.1  c2=0.01  train+val]
  Precision : 0.8347
  Recall    : 0.7365
  F1        : 0.7825

Detailed Classification Report:
              precision    recall  f1-score   support

     Disease       0.83      0.74      0.78       960

   micro avg       0.83      0.74      0.78       960
   macro avg       0.83      0.74      0.78       960
weighted avg       0.83      0.74      0.78       960



---
## Save Results & Comparison Table

In [11]:
os.makedirs('results', exist_ok=True)
results_path = 'results/ner_results.json'

new_result = {
    'method'   : 'crf',
    'precision': round(prec_best, 4),
    'recall'   : round(rec_best,  4),
    'f1'       : round(f1_best,   4)
}

results_list = []
if os.path.exists(results_path):
    with open(results_path) as f:
        try:
            results_list = json.load(f)
        except json.JSONDecodeError:
            pass

updated = False
for r in results_list:
    if r.get('method') == 'crf':
        r.update(new_result)
        updated = True
        break
if not updated:
    results_list.append(new_result)

with open(results_path, 'w') as f:
    json.dump(results_list, f, indent=4)

print('Saved to results/ner_results.json\n')

# ── Comparison table ─────────────────────────────────────────────────────────
print('=' * 58)
print(f'{"Method":<20}  {"Precision":>10}  {"Recall":>8}  {"F1":>8}')
print('-' * 58)
for r in results_list:
    print(f'{r["method"]:<20}  {r["precision"]:>10.4f}  {r["recall"]:>8.4f}  {r["f1"]:>8.4f}')
print('=' * 58)

# Delta vs rule-based
rb = next((r for r in results_list if r['method'] == 'rule_based'), None)
if rb:
    delta = new_result['f1'] - rb['f1']
    print(f'\nCRF vs Rule-Based  Δ F1 = {delta:+.4f}')

Saved to results/ner_results.json

Method                 Precision    Recall        F1
----------------------------------------------------------
rule_based                0.5378    0.6225    0.5771
crf                       0.8347    0.7365    0.7825

CRF vs Rule-Based  Δ F1 = +0.2054
